# DPLQR trajectory diagnostic: no early stopping

This notebook launches the independent command-line implementation in `scripts/run_trajectory.py` and displays its results. The model and original DGP are reused from the repository. Each replicate follows one continuous, unclipped training path; validation/test data never choose weights or change training.

The **intended experiment is 100 repetitions per case and 1,000 epochs**. The editable launcher below defaults to the already supplied two-repetition smoke test. Set `REPETITIONS = 100`, `MAX_EPOCHS = 1000`, `WORKERS = 6`, and `OUTPUT_DIR = EXPERIMENT_DIR` to run/resume the intended experiment. Run cells in order with the repository `.venv` kernel. There is no dependency on the deleted `simulate_homoscedastic.py`.

In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd,*cwd.parents) if (p/'dqAux.py').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('Start this notebook inside the dplqr repository.')
EXPERIMENT_DIR = ROOT/'results/2026-09-08-epoch-trajectory-no-early-stopping'
SCRIPT = EXPERIMENT_DIR/'scripts/run_trajectory.py'
REPETITIONS = 2
MAX_EPOCHS = 50
WORKERS = 3
OUTPUT_DIR = EXPERIMENT_DIR/'smoke-test'
print('Interpreter:',sys.executable)
print('Active profile:',REPETITIONS,'repetitions per case;',MAX_EPOCHS,'epochs')

## Run or resume
The same command works from a terminal; `--repetitions` controls the Monte Carlo design. Existing completed paths are reused only when their settings and source identity match. An interrupted replicate restarts from its fixed seeds. Each worker owns separate data, network and optimizer state.

In [ ]:
command = [sys.executable,str(SCRIPT),'--repetitions',str(REPETITIONS),
           '--max-epochs',str(MAX_EPOCHS),'--workers',str(WORKERS),'--output-dir',str(OUTPUT_DIR)]
with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
    for line in process.stdout:
        print(line,end='')
    result = process.wait()
if result:
    raise subprocess.CalledProcessError(result,command)

## Numerical verification
This checks checkpoint integrity, continuous optimizer steps, finite labeled quantities, Monte Carlo formulas, and independently recomputes selected checkpoint diagnostics from their saved weights and data seeds.

In [ ]:
subprocess.run([sys.executable,str(EXPERIMENT_DIR/'scripts/verify_outputs.py'),str(OUTPUT_DIR)],check=True)

## Results
The Monte Carlo summaries are primary. The preselected individual paths illustrate behavior and are not selected for an interesting effect. L2 error here means the square root of mean squared error; it differs from the paper's relative-MSE table metric.

In [ ]:
display(pd.read_csv(OUTPUT_DIR/'monte_carlo_summary.csv'))
display(Markdown((OUTPUT_DIR/'RESULTS.md').read_text(encoding='utf-8').split('## Figures')[0]))
for figure in sorted((OUTPUT_DIR/'figures').glob('*.png')):
    display(Markdown('### '+figure.stem.replace('_',' ')))
    display(Image(filename=str(figure)))